# Leakage

Data from an experiment design built for a two-level system can be used to fit a *three*-level model, which is how you detect and quantify leakage out of the computational subspace. This page shows how, and how to generate a report whose gate error metrics respect the distinguished role of the first two levels.

## Why a third level

A qubit is usually the lowest two levels of something larger, such as a transmon's anharmonic ladder. When a control pulse has spectral weight on a neighboring transition, population leaves the computational subspace. That is leakage.

Leakage hides information rather than destroying it. Population that leaves can come back, carrying a phase that depends on how long it was gone. A two-level model has nowhere to put that population, hence nowhere to put the memory, and the data ends up matching no single Markovian two-level gate set. Fit three levels and the memory becomes explicit: the leaked amplitude has somewhere to live, and the leaky gate is once again a fixed CPTP map applied identically at every occurrence. What looks non-Markovian in two levels is ordinary Markovian error in three.

Note that this design's fiducials and germs were chosen to amplify two-level parameters, with no guarantee that they amplify every three-level one.

In [1]:
from pygsti.modelpacks import smq1Q_XYI as mp
from pygsti.leakage import leaky_qubit_model_from_pspec, construct_leakage_report
from pygsti.data import simulate_data
from pygsti.protocols import StandardGST, ProtocolData
import numpy as np
import scipy.linalg as la

## A coherent leakage error

`with_leaky_gate` draws a real unit vector $v$ whose first component is zero, forms the rank-one generator $H = s\,vv^{T}$, and composes $U = \exp(iH)$ after the ideal gate. The zero first component is what makes this a leakage error rather than a generic qutrit error: it confines $v$ to levels 1 and 2, so the ground state is untouched and amplitude moves only between the excited computational level and the leakage level. That is the picture for a weakly anharmonic transmon, whose 0-1 drive is detuned from 1-2 by only the anharmonicity. At `strength=0.125` about 0.4% of the population in level 1 leaks per application.

On three levels the error is a unitary: exactly CPTP, exactly Markovian, the same at every occurrence. Restricted to the computational subspace it is not even trace preserving.

In [2]:
def with_leaky_gate(m, gate_label, strength):
    rng = np.random.default_rng(0)
    v = np.concatenate([[0.0], rng.standard_normal(size=(2,))])
    v /= la.norm(v)
    H = v.reshape((-1, 1)) @ v.reshape((1, -1))
    H *= strength
    U = la.expm(1j*H)
    m_copy = m.copy()
    G_ideal = m_copy.operations[gate_label]
    from pygsti.modelmembers.operations import ComposedOp, StaticUnitaryOp
    m_copy.operations[gate_label] = ComposedOp([G_ideal, StaticUnitaryOp(U, basis=m.basis)])
    return m_copy, v

## The target model

`leaky_qubit_model_from_pspec` returns a qutrit *lift* of the qubit model, not a leaky one; the name is a misnomer. Each 2-by-2 gate unitary $u$ is promoted to the 3-by-3 unitary with $u$ in the upper-left block and a 1 in the remaining diagonal entry, so the ideal gates act trivially on the third level. The preparation is $|0\rangle\langle 0|$, and `Mdefault` keeps exactly two effects: $|0\rangle\langle 0|$ for outcome "0", and $|1\rangle\langle 1| + |2\rangle\langle 2|$ for outcome "1". The leakage level is not separately observable; it reads out as a 1. That is why the two-level design's data can be used unchanged.

The default `mx_basis='l2p1'` ("leakage, 2 plus 1") is a Hermitian qutrit basis sorted by that split: four elements span the computational subspace's operators (the qubit identity and Paulis, padded with a zero row and column), one is the projector onto the leakage level, and four span the coherences between them. Hermiticity is required because these models hold real parameter vectors; the `C[...]`/`L[...]` labels matter as much, since pyGSTi reads them to decide that a basis designates a *proper* computational subspace, which switches on the subspace-restricted gauge optimization and report metrics below.

In [3]:
ed = mp.create_gst_experiment_design(max_max_length=8)
tm3 = leaky_qubit_model_from_pspec(mp.processor_spec(), mx_basis='l2p1')
dgm3, leaking_state = with_leaky_gate(tm3, ('Gxpi2', 0), strength=0.125)

## Simulating and fitting

Short circuits carry less information, so `num_samples` is large: $10^5$ shots buy back some of the precision the missing depth would have supplied. That count also outruns pyGSTi's default likelihood regularization: `min_prob_clip` and `radius` both default to $10^{-4}$, while the smallest expected frequency here is $10^{-5}$. The cell below lowers both to $10^{-12}$, which keeps the regularization out of the region where the fit is decided but leaves the objective harder on the optimizer — the fit hits its iteration cap and warns about it at every stage. [Judging fits](../fit/FitStatistics) explains the tradeoff.

In [4]:
num_samples = 100_000
if num_samples > 10_000:
    from pygsti.objectivefns import objectivefns
    objectivefns.DEFAULT_MIN_PROB_CLIP = objectivefns.DEFAULT_RADIUS = 1e-12
ds = simulate_data(dgm3, ed.all_circuits_needing_data, num_samples=num_samples, seed=1997)

The bad-fit machinery acts only when the misfit exceeds `threshold` standard deviations, so setting it to 0.0 makes the `wildcard1d` analysis run for essentially any fit. A wildcard budget is the slack you would have to allow the model's predicted probabilities to make them consistent with the data. The one-dimensional version fits a single scale, spread across gates in proportion to each gate's diamond distance from its target, and the report prints it beside the other error metrics.

In [5]:
gst = StandardGST(
    modes=('CPTPLND',), target_model=tm3, verbosity=4,
    badfit_options={'actions': ['wildcard1d'], 'threshold': 0.0}
)
pd = ProtocolData(ed, ds)
res = gst.run(pd)

-- Std Practice:  Iter 1 of 1  (CPTPLND) --: 
    Precomputing CircuitOutcomeProbabilityArray layouts for each iteration.
      Using MapForwardSimulator without MPI
      Using MapForwardSimulator without MPI
      Using MapForwardSimulator without MPI
      Using MapForwardSimulator without MPI
  --- Iterative GST: Iter 1 of 4  92 circuits ---: 
    --- chi2 GST ---
      --- Outer Iter 0: norm_f = 3.27659e+06, mu=1, |x|=2.92969e-13, |J|=10983.8
      --- Outer Iter 1: norm_f = 2.25253e+06, mu=15206.4, |x|=0.0320479, |J|=390956
      --- Outer Iter 2: norm_f = 197003, mu=15206.4, |x|=0.0758939, |J|=29144.5
      --- Outer Iter 3: norm_f = 9326.78, mu=5068.79, |x|=0.13726, |J|=12106.5
      --- Outer Iter 4: norm_f = 2745.58, mu=4672.67, |x|=0.105771, |J|=12111.7
      --- Outer Iter 5: norm_f = 555.735, mu=3520.97, |x|=0.0926252, |J|=12409.9
      --- Outer Iter 6: norm_f = 209.045, mu=3367.1, |x|=0.088009, |J|=12684.1
      --- Outer Iter 7: norm_f = 175.649, mu=4068.66, |x|=0.08841

      --- Outer Iter 0: norm_f = 157.351, mu=1, |x|=0.0902103, |J|=18167.8
      --- Outer Iter 1: norm_f = 121.581, mu=30940.8, |x|=0.0905852, |J|=18077.1
      --- Outer Iter 2: norm_f = 118.553, mu=12985.5, |x|=0.091457, |J|=18072.2
      --- Outer Iter 3: norm_f = 118.175, mu=13400.8, |x|=0.0930712, |J|=18091.1
      --- Outer Iter 4: norm_f = 117.341, mu=25491.4, |x|=0.0938788, |J|=18088.7
      --- Outer Iter 5: norm_f = 116.966, mu=25711.5, |x|=0.0947696, |J|=18085.6
      --- Outer Iter 6: norm_f = 116.642, mu=194670, |x|=0.0948718, |J|=18085
      --- Outer Iter 7: norm_f = 116.552, mu=105597, |x|=0.0949833, |J|=18084.7
      --- Outer Iter 8: norm_f = 116.467, mu=43371, |x|=0.0951886, |J|=18084.5
      --- Outer Iter 9: norm_f = 116.321, mu=15639.5, |x|=0.0956781, |J|=18084
      --- Outer Iter 10: norm_f = 116.257, mu=33197.1, |x|=0.0963411, |J|=18086.2
      --- Outer Iter 11: norm_f = 116.238, mu=59227.1, |x|=0.0969448, |J|=18086.4
      --- Outer Iter 12: norm_f = 116.08,

  --- Iterative GST: Iter 3 of 4  285 circuits ---: 
    --- chi2 GST ---
      --- Outer Iter 0: norm_f = 270.483, mu=1, |x|=0.110619, |J|=26448.2
      --- Outer Iter 1: norm_f = 220.854, mu=64147.1, |x|=0.110019, |J|=26509.6
      --- Outer Iter 2: norm_f = 219.9, mu=80765.8, |x|=0.109333, |J|=26541.8
      --- Outer Iter 3: norm_f = 218.06, mu=81073.3, |x|=0.109013, |J|=26544
      --- Outer Iter 4: norm_f = 217.816, mu=121132, |x|=0.108784, |J|=26545.5
      --- Outer Iter 5: norm_f = 217.315, mu=143936, |x|=0.108672, |J|=26546.4
      --- Outer Iter 6: norm_f = 216.879, mu=156375, |x|=0.108582, |J|=26539.7
      --- Outer Iter 7: norm_f = 216.871, mu=306717, |x|=0.108516, |J|=26543.9
      --- Outer Iter 8: norm_f = 216.434, mu=306720, |x|=0.108478, |J|=26536.7
      --- Outer Iter 9: norm_f = 216.371, mu=307462, |x|=0.108444, |J|=26537.1
      --- Outer Iter 10: norm_f = 216.338, mu=307957, |x|=0.108412, |J|=26536.1
      --- Outer Iter 11: norm_f = 216.312, mu=307929, |x|=0.108

      --- Outer Iter 1: norm_f = 382.755, mu=237561, |x|=0.108818, |J|=41966.6
      --- Outer Iter 2: norm_f = 378.792, mu=220054, |x|=0.107922, |J|=41948
      --- Outer Iter 3: norm_f = 376.856, mu=220579, |x|=0.107126, |J|=42003.3
      --- Outer Iter 4: norm_f = 374.167, mu=220570, |x|=0.106463, |J|=41946.9
      --- Outer Iter 5: norm_f = 372.978, mu=240514, |x|=0.105891, |J|=41992.2
      --- Outer Iter 6: norm_f = 370.967, mu=241476, |x|=0.105433, |J|=41945
      --- Outer Iter 7: norm_f = 370.113, mu=279611, |x|=0.105018, |J|=41978.6
      --- Outer Iter 8: norm_f = 368.458, mu=280211, |x|=0.104695, |J|=41941.2
      --- Outer Iter 9: norm_f = 367.692, mu=296610, |x|=0.10439, |J|=41962.6
      --- Outer Iter 10: norm_f = 366.702, mu=297979, |x|=0.104125, |J|=41938.4
      --- Outer Iter 11: norm_f = 366.191, mu=320727, |x|=0.103873, |J|=41957.7
      --- Outer Iter 12: norm_f = 365.438, mu=321109, |x|=0.103655, |J|=41936.3
      --- Outer Iter 13: norm_f = 365.029, mu=330416, 

      --- Outer Iter 0: norm_f = 185.304, mu=1, |x|=0.0968529, |J|=30219.5
      --- Outer Iter 1: norm_f = 184.778, mu=325849, |x|=0.0969171, |J|=30122.7
      --- Outer Iter 2: norm_f = 184.61, mu=587284, |x|=0.0968925, |J|=30292.3
      --- Outer Iter 3: norm_f = 183.323, mu=608341, |x|=0.0968976, |J|=30267
      --- Outer Iter 4: norm_f = 182.835, mu=1.44958e+06, |x|=0.0968955, |J|=30354.3
      --- Outer Iter 5: norm_f = 182.484, mu=1.68287e+06, |x|=0.0969012, |J|=30368.9
      --- Outer Iter 6: norm_f = 182.397, mu=2.70219e+06, |x|=0.0969019, |J|=30390
      --- Outer Iter 7: norm_f = 182.101, mu=2.78844e+06, |x|=0.0969051, |J|=30474.1
      --- Outer Iter 8: norm_f = 182.009, mu=3.06071e+06, |x|=0.0969082, |J|=30490
      --- Outer Iter 9: norm_f = 181.975, mu=3.98031e+06, |x|=0.09691, |J|=30500.3
      --- Outer Iter 10: norm_f = 181.935, mu=4.1504e+06, |x|=0.0969112, |J|=30524.4
      --- Outer Iter 11: norm_f = 181.92, mu=1.24512e+06, |x|=0.0969125, |J|=30529.7
      --- Oute

    ******************* Adding Wildcard Budget **************************
    Beginning wildcard budget optimization using alpha bisection method.
      Searching for interval [None, None] with guess 0.1
      Guess value is feasible, 
      Searching for interval [0.1, None] with guess 0.05
      Guess value is feasible, 
      Searching for interval [0.05, None] with guess 0.025
      Guess value is feasible, 
      Searching for interval [0.025, None] with guess 0.0125
      Guess value is feasible, 
      Searching for interval [0.0125, None] with guess 0.00625
      Guess value is feasible, 
      Searching for interval [0.00625, None] with guess 0.003125
      Guess value is feasible, 
      Searching for interval [0.003125, None] with guess 0.0015625
      Guess value is feasible, 
      Searching for interval [0.0015625, None] with guess 0.00078125
      Guess value is feasible, 
      Searching for interval [0.00078125, None] with guess 0.000390625
      Guess value is feasibl

## Reporting

Ordinary gauge optimization minimizes the distance between estimate and target over the gauge group, comparing full 9-by-9 superoperators. That is the wrong objective here: the target's action on the leakage level is a convention we chose, not anything the device was built to do. `construct_leakage_report` compares only the computational block, then finishes with a step restricted to the block-diagonal subgroup that does not mix the computational subspace with the leakage level.

Each estimate in `updated_res` gains a model keyed `LAGO` beside the usual `stdgaugeopt` one, and the two disagree usefully. The error was injected into `Gxpi2` alone; the leakage-aware gauge attributes it there far more sharply than standard gauge optimization does, which pushes more of it onto `Gypi2`. How large the gap looks depends on which fidelity measure you compare them with, but its direction does not. The report's gate tables also gain leakage columns and a switch between subspace-restricted and full-space metrics.

In [6]:
report_dir = '../../../../example_files/leakage-report-automagic'
report_object, updated_res = construct_leakage_report(res, title='easy leakage analysis!')
report_object.write_html(report_dir)

Running idle tomography
Computing switchable properties


<repo>/pygsti/report/reportables.py:1064: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  prob.solve(solver=s)
<repo>/pygsti/report/reportables.py:1064: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  prob.solve(solver=s)
<repo>/pygsti/report/reportables.py:1064: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  prob.solve(solver=s)
<repo>/pygsti/report/reportables.py:1064: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  prob.solve(solver=s)
<repo>/pygsti/report/reportables.py:1064: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more inform

Served with these docs: <a href="../../../../reports/leakage-report-automagic/main.html">leakage-report-automagic</a>.